# CI failure prediction on GitHub Actions - data collection

**Problem.** A CI run on `master` of a large project takes a long time and runs dozens of jobs. If we can predict, *before the run starts*, whether the run will fail, CI time can be prioritised (run likely failures first, notify early) or saved (skip likely passes).

**Research question.** Can a from-scratch, depth-limited CART decision tree, trained on pre-run GitHub Actions and git data, predict CI failures of a large OSS repository better than the history baselines (previous outcome, majority class)? Which pre-run signals drive its decisions?

The assignment requires collecting our own data. This notebook documents how it was collected and what the collected data looks like: coverage over time, run counts, label balance, failure streaks, and re-runs.

## Collection strategy

**Sources.** The targets in `config/targets.json`: repository + workflow pairs, each optionally restricted to one branch. The pooled dataset is analysed as if it came from one CI workflow; every pooled result is also reported per source (target slug). Runs of `push`, `pull_request` and `merge_group` carry a code change and are used. Runs of other events (schedule, dispatch) are listed but get no details.

### Pipeline (per target)

1. **Runs**: `GET /repos/<repo>/actions/workflows/<id>/runs?created=<window>`, plus `branch=` for a target restricted to one branch, one window per month. A window with ≥ 1000 runs (the API cap) is halved, on days and then on hours, until each part fits.
2. **First attempts**: the runs endpoint returns the *latest* attempt. For re-run runs we fetch `attempts/1`: the label is the first-attempt conclusion, which is what the developer saw first.
3. **Jobs**: fetched only for failed first attempts. They feed the previous run's failed-job features and the label check, which drops failures caused only by infrastructure jobs.
4. **Git**: one bare clone per repository, with every branch head and `refs/pull/*/head`, gives every diff, file history and author history without API calls.
5. **Table**: one chronological pass computes the features from runs **completed before** the current run was created. Previous-run features follow the run's lineage (its PR or branch).

Every response is cached on disk, so an interrupted collection resumes where it stopped. The API token is read from `gh auth token` at runtime and never written to disk. Author identities are replaced by salted hashes before any data is saved under `data/processed/`.

In [1]:
from datetime import date

import pandas as pd

from acise_p2_cidt.analysis import labels
from acise_p2_cidt.collect.gh_runs import collect, read_json
from acise_p2_cidt.collect.targets import load_targets, slug
from acise_p2_cidt.data.build import build, write
from acise_p2_cidt.data.load import load, schema_path, table_path
from acise_p2_cidt.viz import plots

# True: label check, every candidate on a 28-day sample.
# False: the real collection, the enabled targets over the window before C.
LABEL_CHECK = True
DEFERRED = {"openclaw__openclaw__209874334"}  # too large for the first pool; not sampled
# Collection date C: everything up to it is development data; runs after it are
# kept for the re-test of the frozen model. Pinned on the day of the full collection.
COLLECTION_UNTIL = date(2026, 11, 4)

if LABEL_CHECK:
    CONFIGS = [c for c in load_targets(enabled_only=False) if slug(c.target) not in DEFERRED]
    SINCE, UNTIL = date(2026, 10, 8), date(2026, 11, 4)
else:
    CONFIGS = load_targets()
    SINCE = (pd.Timestamp(COLLECTION_UNTIL) - pd.DateOffset(months=6)).date()
    UNTIL = COLLECTION_UNTIL
SOURCES = [slug(c.target) for c in CONFIGS]
SOURCES

['n8n-io__n8n__33322401',
 'n8n-io__n8n__33323319@master',
 'astral-sh__ruff__32130301',
 'astral-sh__uv__82881044',
 'windmill-labs__windmill__133951533',
 'windmill-labs__windmill__45664283',
 'sharkdp__fd__11202521',
 'github__gh-aw__180858647',
 'Homebrew__brew__9591',
 'tokio-rs__tokio__1972898',
 'sveltejs__svelte__14783',
 'nushell__nushell__7519865',
 'servo__servo__3876879']

## 1. Collect runs

`LABEL_CHECK = True` collects the label-check sample: every candidate except the deferred ones, 2026-10-08 to 2026-11-04. `False` collects the enabled targets over the 6 months before C. Cached windows cost no calls, so the cell resumes after an interruption. Terminal equivalent: `uv run collect-runs --target <slug> --since <SINCE> --until <UNTIL>`.

In [2]:
COLLECT_RETRIES: int = 10

manifests = {}

for c in CONFIGS:
    display(f"Source: {slug(c.target)}")
    
    for r in range(COLLECT_RETRIES):
        try:
            manifests[slug(c.target)] = read_json(collect(c.target, SINCE, UNTIL))
            break
        except Exception as e:
            display(f"[{r + 1} / {COLLECT_RETRIES}] ERROR: {str(e)}")
        
        if r == COLLECT_RETRIES - 1:
            raise RuntimeError(f"Could not retrieve info about: {slug(c.target)} in {COLLECT_RETRIES} retries!")

'Source: n8n-io__n8n__33322401'

Output()

12538 runs cached, 0 API calls

'Source: n8n-io__n8n__33323319@master'

Output()

1229 runs cached, 0 API calls

... (29 outputs left out)

622 runs cached, 0 API calls

'Source: servo__servo__3876879'

Output()

3319 runs cached, 0 API calls

### Manifests

One row per source: runs, API calls, query windows, the largest window, and runs per event. Each window must stay under the API cap of **1000 runs**, otherwise its tail would be cut silently. The assert is the proof that no window was cut.

In [3]:
manifest_table = (
    pd.DataFrame(
        {
            s: {
                "runs": m["n_runs"],
                "api_calls": m["api_calls"],
                "windows": len(m["windows"]),
                "largest_window": max(m["windows"].values(), default=0),
            }
            | {f"event: {k}": v for k, v in m["runs_per_event"].items()}
            for s, m in manifests.items()
        }
    )
    .T.fillna(0)
    .astype(int)
)
assert (manifest_table["largest_window"] < 1000).all(), manifest_table["largest_window"].idxmax()
manifest_table

,runs,api_calls,windows,largest_window,event: pull_request,event: merge_group,event: workflow_dispatch,event: push,event: workflow_run,event: schedule
n8n-io__n8n__33322401,12538,0,19,963,10312,2184,42,0,0,0
n8n-io__n8n__33323319@master,1229,0,3,662,0,0,0,1229,0,0
astral-sh__ruff__32130301,6198,0,13,976,3185,0,2613,400,0,0
astral-sh__uv__82881044,3170,0,5,964,2728,0,22,420,0,0
windmill-labs__windmill__133951533,2905,0,5,724,0,0,0,2308,597,0
windmill-labs__windmill__45664283,2926,0,5,656,0,0,0,2329,597,0
sharkdp__fd__11202521,106,0,2,97,81,0,0,25,0,0
github__gh-aw__180858647,1665,0,3,803,0,0,0,995,0,670
Homebrew__brew__9591,1029,0,2,726,603,235,0,191,0,0
tokio-rs__tokio__1972898,495,0,2,468,399,0,0,96,0,0


## 1.1 Label check

The label is the first-attempt conclusion. It is only useful if failures come from the code under test. For every registry entry, enabled or not, the check reads the cached runs (no API calls) and reports:

- **infra share**: failures whose failed jobs all match the target's `exclude_jobs`, plus failures without any failed job (the run failed before it tested the code). A pattern matches `<job>` or `<job> :: <failed step>`. These failures are dropped, not relabelled.
- **T2 failures**: failures whose previous run in the same lineage (PR or branch) passed, or that are the first run of their lineage. `est_t2_window` scales the sample count to the 6-month window (only in `LABEL_CHECK` mode).
- **flaky rate**: failed first attempts whose latest attempt passed.

Pass criteria: the workflow builds or tests the project on a supported event, infra share ≤ 30%, estimated T2 failures ≥ 20 for the pool and ≥ 50 for E1/E3.

In [4]:
ALL_CONFIGS = load_targets(enabled_only=False)
summaries = {slug(c.target): labels.summary(c) for c in ALL_CONFIGS}
scale = 182.6 / ((UNTIL - SINCE).days + 1)  # window: (6 months) / sample days

label_table = pd.DataFrame(
    {
        slug(c.target): {
            "enabled": c.enabled,
            "exclude_jobs": len(c.exclude_jobs),
            "failed": s["n_failed"],
            "failure_rate": s["failure_rate"],
            "infra_share": s["infra_share"],
            "no_failed_job": s["n_no_failed_job"],
            "t2_failures": s["t2_failures"],
            "est_t2_window": round(s["t2_failures"] * scale) if LABEL_CHECK else None,
            "t2_share": s["t2_share"],
            "flaky_rate": s["flaky_rate"],
        }
        for c, s in zip(ALL_CONFIGS, summaries.values())
    }
).T
label_table

,enabled,exclude_jobs,failed,failure_rate,infra_share,no_failed_job,t2_failures,est_t2_window,t2_share,flaky_rate
n8n-io__n8n__33322401,True,1,2359,0.251976,0.051717,121,1419,9254,0.634332,0.094955
n8n-io__n8n__33323319@master,True,0,343,0.279772,0.0,0,140,913,0.408163,0.0
astral-sh__ruff__32130301,True,2,580,0.238291,0.255172,124,275,1793,0.636574,0.12069
astral-sh__uv__82881044,True,3,704,0.359918,0.076705,22,360,2348,0.553846,0.274148
windmill-labs__windmill__133951533,True,1,216,0.093709,0.00463,1,103,672,0.47907,0.0
windmill-labs__windmill__45664283,True,0,124,0.053265,0.008065,1,45,293,0.365854,0.0
sharkdp__fd__11202521,True,1,12,0.113208,0.083333,0,9,59,0.818182,0.0
github__gh-aw__180858647,True,1,449,0.617607,0.144766,0,51,333,0.132812,0.0
Homebrew__brew__9591,True,2,99,0.109031,0.111111,5,60,391,0.681818,0.080808
openclaw__openclaw__209874334,False,0,0,NaN,NaN,0,0,0,NaN,NaN


In [5]:
# Failed jobs left after exclusion, ranked until they cover >= 80% of the failures left
for s, x in summaries.items():
    if len(x["top_jobs"]):
        display(s, x["top_jobs"])

'n8n-io__n8n__33322401'

,job,failed_step,n_runs,coverage
0,Unit tests / Unit tests,Fail if tests failed,1057,0.472508
1,Typecheck,Setup Node.js,661,0.646848
2,DB Tests / Postgres 18 Integration,Run Integration Tests,234,0.682164
3,Unit tests / CLI Unit Tests (2/2),"Test Unit (cli, scoped)",204,0.682164
4,Install & Build,Setup and Build,198,0.770675
5,Unit tests / Frontend (4/4),Test,190,0.770675
6,DB Tests / SQLite Pooled Integration,Run Integration Tests,172,0.782298
7,Unit tests / Backend Unit Tests,"Test Unit (backend, except cli and nodes-base)",157,0.782298
8,Unit tests / CLI Integration Tests (2/2),"Test Integration (cli, scoped)",154,0.782298
9,Format check,Run format check,127,0.824318


'n8n-io__n8n__33323319@master'

,job,failed_step,n_runs,coverage
0,Verify single-instance deps (npm-install) / ve...,Verify npm-install closure (changed packages),172,0.501458
1,Unit tests (24.18.1) / Unit tests,Fail if tests failed,91,0.760933
2,Unit tests (24.22.1) / Unit tests,Fail if tests failed,73,0.900875


'astral-sh__ruff__32130301'

,job,failed_step,n_runs,coverage
0,cargo test (namespace-profile-macos-15),Run tests,149,0.344907
1,cargo hawk,Hawk,124,0.537037
2,"cargo test (linux, release)",Run tests,95,0.553241
3,cargo test (linux),Run tests,94,0.557870
4,cargo test (namespace-profile-windows-2022-x86...,Run tests,82,0.645833
5,cargo clippy,Clippy,41,0.687500
6,cargo publish dry-run,cargo publish dry-run,29,0.717593
7,Fuzz for new ty panics/timeouts,Fuzz,26,0.743056
8,cargo build (msrv),<no failed step>,21,0.784722
9,cargo publish dry-run,<no failed step>,20,0.791667


... (16 outputs left out)

'nushell__nushell__7519865'

,job,failed_step,n_runs,coverage
0,Linux / root,Test,5,0.172414
1,Linux / nu-path / fuzz,Fmt,4,0.310345
2,Linux / root,Check,4,0.448276
3,Windows / root,Test,4,0.586207
4,Linux / nu-parser / fuzz,Assert Clean Repo,2,0.655172
5,Linux / nu-path / fuzz,<no failed step>,2,0.724138
6,Linux / root,Assert Clean Repo,2,0.724138
7,Linux / root,Clippy,2,0.793103
8,Linux / root,Fmt,2,0.793103
9,WASM / root,Assert Clean Repo,2,0.793103


'servo__servo__3876879'

,job,failed_step,n_runs,coverage
0,Lint / Lint / Lint,Tidy,277,0.319493
1,Lint / Lint / Lint,Clippy,222,0.575548
2,Linux (Unit Tests) / Linux / Call common Linux...,Build (checked-release),194,0.607843
3,"Linux (Unit Tests, Build libservo, WPT, Benche...",Run tests,109,0.733564
4,Linux (Unit Tests) / Linux / Call common Linux...,Unit tests,42,0.761246
5,"Linux (Unit Tests, Build libservo, WPT, Benche...",Merge logs (filtered),37,0.802768


## 2. Build the per-run table

Joins the API cache with the git history (a full clone of `master` is made under `data/raw/` on the first build, which takes a few minutes). Only `success` and `failure` first attempts are kept; every other conclusion is dropped and counted. The table is built only if it does not exist yet.

In [ ]:
if not table_path(TARGET_REPO.repo).exists():
    write(*build(TARGET_REPO), TARGET_REPO)

schema = read_json(schema_path(TARGET_REPO.repo))
df = pd.read_csv(table_path(TARGET_REPO.repo), parse_dates=["created_at"])

print(f"{schema['n_runs']} runs, {schema['n_failed']} failed, {len(schema['feature_cols'])} features")
print("collected at:", schema["collected_at"])
print("range:", df["created_at"].min(), "to", df["created_at"].max())

In [ ]:
outcomes = pd.Series(
    {"success": schema["n_runs"] - schema["n_failed"], "failure": schema["n_failed"]}
    | {f"dropped: {k}": v for k, v in schema["n_dropped"].items()},
    name="runs",
)
outcomes.to_frame().assign(share=lambda d: (d["runs"] / d["runs"].sum()).round(4))

## 3. Coverage over time

Runs and failure rate per month. Gaps would mean missing windows; a moving failure rate means drift that the time-ordered evaluation has to cope with.

In [ ]:
df.groupby(df["created_at"].dt.strftime("%Y-%m"))["failed"].agg(
    runs="size", fail_rate="mean"
).round(3).T

Between February and May 2026 the failure rate jumps from ~10–15% to 43–72%. We call this period the **regime** (2026-02-10 -> 2026-06-10, read from the weekly plot below). It is kept in the data (it is real CI behaviour); the tree notebook adds a sensitivity check that repeats the CV without it.

In the weekly plot, bars are runs and the line is the failure rate. The shaded band is the regime, and the dashed line marks the end of the **dev segment** (first 20% of runs, used only for EDA and feature selection).

In [ ]:
REGIME = (pd.Timestamp("2026-02-10", tz="UTC"), pd.Timestamp("2026-06-10", tz="UTC"))
in_regime = df["created_at"].between(*REGIME, inclusive="left")

t1, t2 = load(TARGET_REPO.repo, "T1"), load(TARGET_REPO.repo, "T2")
dev_end = t1.order.iloc[t1.meta["n_dev"]]
print("dev segment ends:", dev_end, f"(fail rate {t1.y[t1.dev_mask].mean():.1%})")

fig = plots.runs_per_week(df)
fig.add_vrect(x0=REGIME[0], x1=REGIME[1], fillcolor="red", opacity=0.08, line_width=0)
fig.add_vline(x=dev_end, line_dash="dash")
fig.update_layout(yaxis2_tickformat=".0%", legend={"orientation": "h", "y": 1.1})
plots.save(fig, "collection_runs_per_week")

In [ ]:
display(
    df.groupby(in_regime.map({True: "regime", False: "outside"}))["failed"].agg(
        runs="size", failed="sum", fail_rate="mean"
    )
)
# Diagnostic only: job names are repository-specific and never used as features.
f = df[df["failed"] == 1]
node25 = f["failed_job_names"].fillna("").str.contains("(25.x)", regex=False)
r = in_regime.loc[f.index]
print(f"share of all failures inside the regime: {r.mean():.1%}")
print(f"failed runs with a Node 25.x job failing: regime {node25[r].mean():.1%}, outside {node25[~r].mean():.1%}")

## 4. Label balance - T1 vs. T2

- **T1**: all runs.
- **T2**: only runs whose previous completed run passed (*first* failures). A failure that follows a failure is easy to guess (Parrot baseline); T2 removes those and keeps the harder case.

If T2 had fewer than ~300 failures, T1 would become the main target.

In [ ]:
labels = pd.concat(
    [pd.DataFrame({"failed": ds.y, "target": name}) for name, ds in [("T1", t1), ("T2", t2)]]
)
display(labels.groupby("target")["failed"].agg(runs="size", failed="sum", fail_rate="mean"))
plots.save(plots.label_balance(labels, by="target"), "collection_label_balance")


**Run retention.** GitHub deletes workflow runs after about 400 days and updates their `updated_at` shortly before. The oldest 355 cached runs were already marked this way at collection time (one of them already returns 404), so their completion time is taken from their jobs or estimated (`MAX_RUN_SECS` in `data/build.py`). The raw cache is the only remaining copy of these runs.

## 5. Failure streaks

Length of consecutive-failure streaks (in creation order, log scale). Long streaks mean a broken `master` stays broken over several pushes. That is what makes the previous outcome a strong baseline on T1, and the reason T2 exists.

In [ ]:
s = df["failed"]
streaks = s.groupby((s != s.shift()).cumsum()).agg(["first", "size"])
fail_streaks = streaks.loc[streaks["first"] == 1, "size"]

print(fail_streaks.describe().round(4).to_dict())
print(f"failures that follow a failure: {(fail_streaks - 1).sum() / s.sum():.4%}")

plots.save(
    plots.streak_hist(fail_streaks, nbins=int(fail_streaks.max()), log_y=True),
    "collection_fail_streaks",
)

## 6. Re-runs and flaky share

A run is **flaky** when its first attempt failed and a later attempt of the same commit passed. Nothing in the code changed, so such failures cannot be predicted from the change itself.

Re-runs are almost never used on n8n's `master`: a failure is fixed by a new commit, not retried. The flaky share measured from re-runs is therefore a **lower bound** and says little about how much of the label is noise. `was_rerun` and `flaky` are diagnostics only and never used as features (they are known only after the run is completed).

In [ ]:
failed = df[df["failed"] == 1]
display(pd.crosstab(df["failed"], df["conclusion_latest"], margins=True))

print(f"re-run share: {df['was_rerun'].mean():.4%}")
print(f"failures re-run: {failed['was_rerun'].mean():.4%}")
print(f"flaky share of failures: {failed['flaky'].mean():.4%}")

## Observations

**Size and coverage.** 10,277 `push` runs of `CI: Master` were collected (57 query windows, the largest with 748 runs, so none hit the 1000-run API cap). The data covers 2025-09-16 -> 2026-11-02 (collection date C). After filtering, **10,265 runs** remain (7,565 passed, 2,700 failed). Only 12 runs were dropped (9 cancelled, 1 startup failure, 2 still queued or running at C). 5 runs point to a commit no longer on `master`; their git features are NaN.

**Volume grows.** Weekly runs grow from ~100 (late 2025) to 250–400 (autumn 2026). CI-load features (`ci_load_*`, `concurrent_runs`) drift with it.

**A failure regime (Feb–May 2026).** The monthly failure rate is 8–14% before February 2026 and 11–26% after June 2026, but 43–72% between. That period holds 2,533 runs and **1,471 failures (54% of all failures)**. In over 80% of its failed runs, a Node 25.x unit-test job failed. These failures come from the CI environment (a Node version leg that kept failing), not from the pushed change. History features predict them trivially, so they inflate T1 results for both the tree and the Parrot baseline. The regime is **kept**, since it is real CI behaviour; the tree notebook repeats the CV without it as a sensitivity check, and T2 (less affected: 42% failure rate inside vs. 12% outside) is the main target.

**Dev segment.** The first 20% of runs end on 2026-02-02, just before the regime; their failure rate is 12%. Feature selection therefore sees only the normal period, while the first CV folds fall inside the regime.

**T1 vs. T2.** T1 has 10,265 runs and 2,700 failures (26.3%). T2 has 7,302 runs and 1,213 failures (16.6%), well above the 300-failure threshold.

**Failures cluster.** There are 1,169 failure streaks. The median is 1 and the mean 2.3, with a long tail up to 46 runs. 56.7% of failures directly follow a failure, which makes the previous outcome (Parrot) a strong baseline on T1; T2 removes exactly these.

**Re-runs.** Only 0.14% of runs were re-run (0.52% of failures), and 9 failures later passed on a re-run (0.33%). Failures on `master` are fixed by new commits, not retried, so the flaky share cannot be measured from re-runs: 0.33% is a lower bound only.